# End-to-End AI Audio Processing & Speaker-Aware Subtitle Generation

  

---

### Project Overview
This notebook presents a fully automated, machine-learning pipeline designed to process raw video content and generate highly accurate, speaker-attributed subtitles (SRT). It bridges the gap between raw unstructured audio and structured, timestamped text by fusing state-of-the-art automatic speech recognition (ASR) with advanced speaker diarization.

### Core Objectives & Workflow
1. **Media Acquisition:** Dynamically extract and format high-quality audio/video from YouTube.
2. **Acoustic Purification:** Isolate pure human vocals from background noise/music using source separation models.
3. **Speech-to-Text:** Generate granular, word-level transcripts using optimized ASR.
4. **Speaker Diarization:** Build a precise timeline of speaker turns and extract dynamic voice profiles.
5. **Data Fusion:** Algorithmically align the text timestamps with the speaker timeline to attribute every word to a specific person.
6. **Deliverable Generation:** Chunk the fused data into an industry-standard, readable `.srt` subtitle file.

### Technology Stack
* **ASR:** `faster-whisper` (CTranslate2 optimized)
* **Diarization:** `pyannote.audio`
* **Source Separation:** `demucs`
* **Media Processing:** `ffmpeg`, `yt-dlp`, `pydub`
* **Deep Learning Framework:** `PyTorch`

---

### Note on Documentation Standards
*Per the project requirements, code is not the only deliverable in this notebook.* Before every major execution cell, you will find detailed documentation breaking down the task. Each section explicitly highlights:
* **Outputs:** What the block of code successfully generated.
* **Challenges:** The technical hurdles associated with that specific step in an AI pipeline.
* **Engineering Choices:** The specific reasoning behind the algorithms, hyperparameter tuning, and library selections used to overcome those challenges.

## 1. Environment Setup & Dependency Installation

**Outputs:** Successfully installed all core libraries required for the pipeline: audio extraction (`yt-dlp`), speech-to-text (`faster-whisper`), speaker diarization (`pyannote.audio`), and audio preprocessing (`demucs`, `pydub`), alongside their foundational deep learning frameworks (`torch`, `transformers`).

**Challenges:**
Setting up audio-AI environments often leads to "dependency hell," particularly with conflicting versions of NumPy, PyTorch, and HuggingFace Transformers.

**Engineering Choices:**
* **Version Pinning:** I explicitly pinned `torch==2.8.0`, `transformers==4.40.0`, and forced `"numpy<2.0"`. `pyannote.audio` is known to break on Numpy 2.0+, and locking the deep learning libraries ensures the environment is reproducible and won't suddenly crash if a library releases a breaking update tomorrow.
* **`faster-whisper` over `whisper`:** I opted for the CTranslate2 implementation of Whisper (`faster-whisper`) instead of OpenAI's default package because it drastically reduces GPU memory usage and executes significantly faster, which is critical when running resource-heavy notebooks.

In [ ]:
!pip install yt-dlp -q
!pip install faster-whisper -q
!pip install pyannote.audio "numpy<2.0" -q
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 -q
!pip install transformers==4.40.0 huggingface-hub==0.22.2 -q
!pip install -q demucs pydub

### 2. Library Imports & Workspace Configuration

**Engineering Choices:**
* **Comprehensive Tooling:** I imported standard Python utilities (`os`, `json`, `glob`, `subprocess`) to handle the heavy file management and formatting required later, alongside the specialized ML/Audio libraries (`faster_whisper`, `pyannote`, `pydub`, `torch`).


In [ ]:
import os
import json
import glob
import subprocess
import warnings
import yt_dlp
import torch
import pandas as pd
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
from pyannote.core import Annotation
from huggingface_hub import login
from pydub import AudioSegment

# Suppress warnings to keep output clean
warnings.filterwarnings("ignore")

## 3. Workspace Cleanup & Media Acquisition

**Outputs:** Cleared previous run artifacts (old transcripts) to reset the environment, and successfully downloaded the target media. Merged the highest quality audio and video streams into a standardized `source_video.mp4` file.


In [ ]:
file_path = '/kaggle/working/transcript_with_timestamps.json'
if os.path.exists(file_path):
    os.remove(file_path)
    print("Old file deleted successfully.")
else :
      print("File not found.")

# 1. Put your test video link here
video_url = "https://www.youtube.com/watch?v=L6G0VxscwqE"

# 2. Options for yt-dlp
ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
    'outtmpl': 'source_video.%(ext)s', 
    'quiet': False 
}

# 3. Execute the download
print("Starting download...")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])
print("✅ Download Complete! Check your files.")

## 4. Audio Transcription & Word-Level Timestamping

**Outputs:** Successfully transcribed the entire audio file, correctly identifying the language (English), and saved the output as `transcript_with_timestamps.json`. The JSON contains granular timing data (start and end times) for every single word spoken.


**Engineering Choices:**
* **Memory Optimization:** I instantiated the `WhisperModel` using `device="cuda"` to leverage hardware acceleration, but crucially added `compute_type="float16"`. This halves the memory footprint of the model weights without noticeably sacrificing transcription accuracy, ensuring the notebook runs stably.
* **Granular Data Extraction:** I set `word_timestamps=True` during the `.transcribe()` call. This forces the model to calculate the exact millisecond each word begins and ends. This granular data structure is strictly necessary for the upcoming "fusion" step, where these words will be mapped to Pyannote's speaker timeline.

In [ ]:
video_file_path = "source_video.mp4" 

print("Loading the AI model into the GPU...")
model = WhisperModel("base", device="cuda", compute_type="float16")

print(f"Listening to {video_file_path}...")
segments, info = model.transcribe(video_file_path, word_timestamps=True)
print(f"Detected language: {info.language} with {info.language_probability:.2f}% probability\n")

transcript_data = []
for segment in segments:
    for word in segment.words:
        word_data = {
            "word": word.word.strip(),
            "start": round(word.start, 2),
            "end": round(word.end, 2)      
        }
        transcript_data.append(word_data)
        print(f"[{word_data['start']}s -> {word_data['end']}s] {word_data['word']}")

output_filename = "transcript_with_timestamps.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(transcript_data, f, indent=4, ensure_ascii=False)
print(f"\n✅ Success! Saved all timestamps to {output_filename}")

## 5. Audio Preprocessing & Vocal Isolation

**Outputs:** Successfully extracted the raw audio into a standardized format (`raw_audio.wav`) and utilized an AI stem-separator to isolate pure human vocals (`vocals.wav`), saving them for the diarization step.

**Challenges:** 
1. **PyTorch Security Updates:** Recent versions of PyTorch restrict loading certain pickled weights by default (`weights_only=True`), which completely breaks many open-source models that rely on older serialization methods.

2. **Acoustic Interference:** Diarization models (like Pyannote) are highly sensitive to background noise, music, or environmental sounds, which cause them to "hallucinate" non-existent speakers.

**Engineering Choices:**
* **Notebook-Safe PyTorch Patch:** Instead of downgrading PyTorch, I engineered a monkey-patch to temporarily override the `torch.load` function, setting `weights_only=False`. This safely bypasses the security restriction in the runtime while allowing the models to load correctly.
* **Standardized `ffmpeg` Parameters:** I explicitly configured `ffmpeg` to downmix the audio to mono (`-ac 1`) and resample it to 16kHz (`-ar 16000`). This is the mathematical baseline expected by almost all speech-based neural networks; feeding them 44.1kHz stereo audio degrades their accuracy.
* **Demucs Preprocessing:** To prevent AI hallucinations, I ran the `htdemucs` model via a subprocess, specifically requesting a two-stem separation (`--two-stems vocals`). By physically stripping out everything except human speech before feeding the audio to Pyannote, I dramatically increased the precision of the speaker detection.

In [ ]:
if not hasattr(torch, "_is_patched"):
    _original_load = torch.load
    def _unlocked_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)
    torch.load = _unlocked_load
    torch._is_patched = True 
try:
    torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])
except AttributeError:
    pass

print("1️⃣ Extracting raw audio...")
os.system("ffmpeg -i source_video.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 raw_audio.wav -y -loglevel error")

print("2️⃣ Isolating vocals to prevent AI hallucinations...")
subprocess.run(
    ["python", "-m", "demucs.separate", "-n", "htdemucs", "--two-stems", "vocals", "-d", "cuda", "raw_audio.wav"], 
    check=True
)

clean_vocal_path = glob.glob("separated/*/raw_audio/vocals.*")[0]
print(f"✅ Found pure vocals at: {clean_vocal_path}")

## 6 Speaker Diarization & Hallucination Scrubbing

**Outputs:** Authenticated securely with HuggingFace, successfully loaded the `pyannote/speaker-diarization-3.1` model into GPU memory, and generated a highly accurate, timeline-mapped list of speaker turns (`SPEAKER_00` and `SPEAKER_01`).

**Challenges:** 
1. **Access Control:** State-of-the-art diarization models like Pyannote 3.1 are gated and require secure token authentication to download the weights.
2. **Over-segmentation & Thrashing:** Out-of-the-box diarization models frequently suffer from "speaker thrashing"—rapidly alternating between speakers or creating dozens of tiny, incorrect segments due to breathing, lip smacks, or micro-noises.

**Engineering Choices:**
* **Hyperparameter Tuning:** Instead of accepting the default model parameters, I passed a custom configuration to the pipeline (`clustering: { threshold: 0.60 }`). This specifically tunes the model's clustering sensitivity, forcing it to be mathematically more confident before it decides a new speaker has taken over, which drastically reduces false positives.
* **Programmatic Scrubbing:** I built a secondary filtering loop that iterates through the raw diarization output and drops any speech segment shorter than 0.2 seconds (`turn.end - turn.start > 0.2`). This acts as a final safety net to scrub out micro-artifacts, ensuring the timeline is clean and ready to be merged with the Whisper text data.

In [ ]:
print("\n3️⃣ Loading Pyannote model into memory...")
try:
    # Authenticate with HuggingFace
    login("HF_TOKEN")
    
    pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
    pipeline.to(torch.device("cuda"))
    
    # HYPERPARAMETER TUNING
    pipeline.instantiate({
        "clustering": {
            "threshold": 0.60  
        }
    })
    
    print("Listening to PURIFIED audio with tuned sensitivity...")
    raw_diarization = pipeline(
        clean_vocal_path, 
        min_speakers=1, 
        max_speakers=5 
    )

    print("Scrubbing AI hallucinations and speaker thrashing...")
    diarization = Annotation()
    
    print("\n✅ Purified Speaker Timeline:")
    for turn, _, speaker in raw_diarization.itertracks(yield_label=True):
        if turn.end - turn.start > 0.2:
            diarization[turn] = speaker
            print(f"[{turn.start:.2f}s -> {turn.end:.2f}s] {speaker}")
            
except Exception as e:
    print(f"\n❌ Error details: {e}")

## 7. Dynamic Voice Profile Extraction

**Outputs:** Successfully generated and saved pristine reference audio files (`ref_SPEAKER_00.wav`, `ref_SPEAKER_01.wav`) containing the highest-quality speech sample for each unique acoustic profile detected in the video.

**Challenges:** If this pipeline is extended for downstream AI tasks like Zero-Shot Voice Cloning or speaker verification, it requires clean reference audio. Grabbing the *first* timestamp a speaker appears is dangerous, as it might just be a 0.3-second grunt or a heavily interrupted word.

**Engineering Choices:**
* **Algorithmic Selection:** We engineered a search loop that iterates through the entire diarization timeline to specifically find the *longest continuous block* of speech for each speaker. This guarantees the highest quality, most representative sample of their voice.
* **Duration Capping (Memory Optimization):** We implemented a hard ceiling of 6 seconds (6000ms) for the reference clips. Most state-of-the-art voice models only require 3 to 6 seconds to capture an acoustic profile; anything longer wastes disk space and slows down downstream processing.
* **Source Purification:** Crucially, We sliced these timestamps from the `clean_vocals` object (the Demucs output) rather than the raw `.mp4` audio. This ensures the saved profiles are 100% pure human voice, completely free of the original background noise.

In [ ]:
print("Loading purified audio into memory...")
clean_vocal_path = glob.glob("separated/*/raw_audio/vocals.*")[0]
clean_vocals = AudioSegment.from_file(clean_vocal_path)

print("\n🔍 Building dynamic voice references for all detected speakers...")
unique_speakers = list(set([speaker for _, _, speaker in raw_diarization.itertracks(yield_label=True)]))
print(f"Detected {len(unique_speakers)} unique acoustic profiles.")

for target_speaker in unique_speakers:
    longest_duration = 0
    best_start = 0
    best_end = 0
    
    for turn, _, speaker in raw_diarization.itertracks(yield_label=True):
        if speaker == target_speaker:
            duration = turn.end - turn.start
            if duration > longest_duration:
                longest_duration = duration
                best_start = turn.start
                best_end = turn.end
    
    start_ms = int(best_start * 1000)
    end_ms = int(best_end * 1000)
    
    if (end_ms - start_ms) > 6000:
        end_ms = start_ms + 6000
        
    ref_filename = f"ref_{target_speaker}.wav"
    clean_vocals[start_ms:end_ms].export(ref_filename, format="wav")
    
    actual_duration = (end_ms - start_ms) / 1000
    print(f"  -> Saved {ref_filename} (Length: {actual_duration:.2f}s)")

## 8. Data Fusion: Aligning Whisper Text with Pyannote Speakers

**Outputs:** Successfully fused the output of two distinct AI models into a single, comprehensive dataset (`final_master_transcript.json`). Every individual spoken word is now precisely mapped to its corresponding speaker ID without any "UNKNOWN" gaps.

**Challenges:** Whisper and Pyannote process audio entirely differently. Whisper outputs specific timestamps for *words*, while Pyannote outputs timestamps for *continuous blocks of speech*. Because human speech is messy (people overlap, breathe, or pause mid-word), these two timelines rarely perfectly align mathematically, resulting in "UNKNOWN" speaker tags for words that fall in the gaps.

**Engineering Choices:**
* **Midpoint Calculation:** Instead of checking if a word's start or end time falls inside a speaker block, I calculated the mathematical *midpoint* of each word (`(word_start + word_end) / 2.0`). This is a much more robust anchor point for matching across timelines.
* **Padding Buffers:** I added a `+/- 0.2` second buffer to the Pyannote speaker blocks during the comparison step to account for minor synchronization drift between the two models.
* **Stateful Fallback Logic:** To completely eradicate "UNKNOWN" words (which break downstream applications like SRT generation), I implemented a fallback variable (`last_known_speaker`). If a word falls into a true dead zone between Pyannote blocks, the algorithm intelligently assigns it to the person who was most recently talking, assuming a continuous sentence.

In [ ]:
video_file_path = "source_video.mp4" 

print("Loading Whisper to grab the text...")
whisper_model = WhisperModel("base", device="cuda", compute_type="float16")
segments, info = whisper_model.transcribe(
    video_file_path, 
    word_timestamps=True,
    condition_on_previous_text=False 
)

print("Merging words with your speaker labels...")
final_pipeline_data = []
last_known_speaker = "SPEAKER_00"

for segment in segments:
    for word in segment.words:
        word_start = round(word.start, 2)
        word_end = round(word.end, 2)
        word_midpoint = (word_start + word_end) / 2.0
        
        current_speaker = "UNKNOWN"
        
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            if (turn.start - 0.2) <= word_midpoint <= (turn.end + 0.2):
                current_speaker = speaker
                last_known_speaker = speaker
                break 
                
        if current_speaker == "UNKNOWN":
            current_speaker = last_known_speaker
                
        word_data = {
            "word": word.word.strip(),
            "start": word_start,
            "end": word_end,
            "speaker": current_speaker
        }
        final_pipeline_data.append(word_data)
        print(f"[{word_start}s -> {word_end}s] {current_speaker}: {word_data['word']}")

output_filename = "final_master_transcript.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_pipeline_data, f, indent=4, ensure_ascii=False)
print(f"\n✅ Merge Complete! No more UNKNOWNs. Saved to {output_filename}")

## 9. Final Deliverable: Subtitle (SRT) Generation

**Outputs:** Successfully transformed the fused JSON word data into an industry-standard SubRip Subtitle (`.srt`) file, formatting the text and timestamps for immediate video playback.

**Challenges:** Taking a raw array of word-level timestamps and dumping them directly into a subtitle file creates an unwatchable experience (words flashing on screen one by one). Conversely, clumping too much text together ignores natural conversational flow. 

**Engineering Choices:**
* **Precise Timestamp Formatting:** I utilized `pandas.to_timedelta` to handle the complex modulo arithmetic required to accurately convert raw floating-point seconds into the strict `HH:MM:SS,mmm` string format mandated by the SRT specification.
* **Intelligent Chunking Algorithm:** Instead of just grouping sentences by arbitrary word counts, I engineered a dynamic grouping loop that splits the subtitles based on actual audio context. A new subtitle block is forced under two conditions:
  1. **Speaker Change:** The moment a different person begins talking, a new line is created.
  2. **Silence Gap:** The algorithm calculates the delta between the current word and the previous word (`w["start"] - prev_w["end"]`). If that gap exceeds 1.5 seconds, it forces a new block. This ensures the text on screen naturally mimics the real-life pacing and pauses of the conversation.

In [ ]:
def format_timestamp(seconds: float):
    """Converts seconds to SRT timestamp format (HH:MM:SS,mmm)"""
    td = pd.to_timedelta(seconds, unit='s')
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    millis = int(td.microseconds / 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"

def generate_srt(word_data, output_file):
    if not word_data:
        return

    segments = []
    current_seg = {
        "speaker": word_data[0]["speaker"],
        "start": word_data[0]["start"],
        "words": [word_data[0]["word"]]
    }

    # Group words into sentences based on speaker changes or pauses > 1.5s
    for i in range(1, len(word_data)):
        w = word_data[i]
        prev_w = word_data[i-1]
        
        # Start a new subtitle line if speaker changes OR there is a long silence
        if w["speaker"] != current_seg["speaker"] or (w["start"] - prev_w["end"] > 1.5):
            current_seg["end"] = prev_w["end"]
            current_seg["text"] = " ".join(current_seg["words"])
            segments.append(current_seg)
            
            current_seg = {
                "speaker": w["speaker"],
                "start": w["start"],
                "words": [w["word"]]
            }
        else:
            current_seg["words"].append(w["word"])
            
    # Add the last segment
    current_seg["end"] = word_data[-1]["end"]
    current_seg["text"] = " ".join(current_seg["words"])
    segments.append(current_seg)

    # Write to File
    with open(output_file, 'w', encoding='utf-8') as f:
        for i, seg in enumerate(segments, start=1):
            start_time = format_timestamp(seg['start'])
            end_time = format_timestamp(seg['end'])
            f.write(f"{i}\n")
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"[{seg['speaker']}]: {seg['text']}\n\n")

# Execute the creation using the data from Cell 8
generate_srt(final_pipeline_data, "final_subtitles.srt")
print("✅ Subtitles successfully generated: final_subtitles.srt")